In [1]:
import kagglehub
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Download latest version
path = kagglehub.dataset_download("csafrit2/maternal-health-risk-data")

print("Path to dataset files:", path)

# List files to find the CSV
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
data_path = os.path.join(path, csv_file)

df = pd.read_csv(data_path)
df.head()

/Users/rogeriobrumhermany/Projects/pos/pregnancy-health-helper/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/rogeriobrumhermany/Projects/pos/pregnancy-health-helper/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|                                                                                                                                                                         | 0.00/3.77k [00:00<?, ?B/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3.77k/3.77k [00:00<00:00, 1.98MB/s]

Extracting files...


Path to dataset files: /Users/rogeriobrumhermany/.cache/kagglehub/datasets/csafrit2/maternal-health-risk-data/versions/1


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


In [2]:
# Preprocessing
print("Risk levels:", df['RiskLevel'].unique())

# Map RiskLevel to numeric values
risk_mapping = {'low risk': 0, 'mid risk': 1, 'high risk': 2}
df['RiskLevel_encoded'] = df['RiskLevel'].map(risk_mapping)

X = df.drop(['RiskLevel', 'RiskLevel_encoded'], axis=1)
y = df['RiskLevel_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Risk levels: ['high risk' 'low risk' 'mid risk']
Training set shape: (811, 6)
Testing set shape: (203, 6)


In [3]:
# Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=risk_mapping.keys()))

Accuracy: 0.8177339901477833

Classification Report:
               precision    recall  f1-score   support

    low risk       0.86      0.76      0.81        80
    mid risk       0.75      0.84      0.80        76
   high risk       0.87      0.87      0.87        47

    accuracy                           0.82       203
   macro avg       0.83      0.83      0.83       203
weighted avg       0.82      0.82      0.82       203



In [4]:
# Save Model
model_path = os.path.join(os.getcwd(), 'maternal_health_model.joblib')
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to /Users/rogeriobrumhermany/Projects/pos/pregnancy-health-helper/training/maternal_health_model.joblib
